# GPA778 - Laboratory 2
# Part C - Perception III: ROS2 Actions and Point Cloud Filtering

During the last laboratory, you implemented a 3D scan system with a servo-mounted 2D LiDAR, controlled by **ROS2 services**. Today, you will discover **ROS2 actions**, a mechanism better suited for long-running operations, and learn how to filter point clouds.

**Laboratory Objectives:**
- Understand the difference between ROS2 services and actions
- Understand the role of ROS2 executors and callback groups
- Integrate a state machine into an action's execution callback
- Integrate a processing state into the state machine
- Implement point cloud filters with Open3D
- Configure filtering parameters via a ROS2 action

**Estimated Duration:** 2h30

---
## Section 1: ROS2 Services Review

### What is a ROS2 Service?

A ROS2 service is a **synchronous** **request-response** communication mechanism.

**Characteristics:**
- The client sends a request and **waits** for the response
- Blocking for the client during the entire execution duration
- Suitable for **short** operations (<1 second)

**Example from Lab 2b: SetServoPos**

You used the `SetServoPos` service to control the servo position:

```python
# Create the client
self.set_servo_pos_srv = self.create_client(SetServoPos, 'set_servo_pos')

# Send a request
req = SetServoPos.Request()
req.position = 2000
future = self.set_servo_pos_srv.call_async(req)
future.add_done_callback(self._on_servo_response)
```

### Limitations of Services

**Reflection Question:**
> "What happens if a service takes 10 seconds to execute? Can the client know the execution progress?"

**Answer:** No! The client waits in silence until receiving the final response. It has **no feedback** during execution.

---
## Section 2: Introduction to ROS2 Actions

### The Problem with Services for Long-Running Operations

Imagine a 3D scan that takes **8 seconds** to execute:
- The client is **blocked** for 8 seconds
- Impossible to know if the scan is at 10%, 50%, or 90%
- Impossible to **cancel** the ongoing scan
- Impossible to know if the server has crashed or is simply slow

**Solution: ROS2 Actions**

### What is a ROS2 Action?

An action is an **asynchronous** communication mechanism for **long-running** operations featuring:
- **Periodic feedback** during execution
- Ability to **cancel**
- Final result at completion

### Service vs Action Comparison

| Aspect | Service | Action |
|--------|---------|--------|
| Duration | Short (<1s) | Long (>1s) |
| Feedback | ❌ No | ✅ Yes (periodic) |
| Cancellation | ❌ No | ✅ Yes |
| Result | Immediate | Deferred |
| Client Blocking | ✅ Blocking | ❌ Asynchronous |
| Example | `SetServoPos` | `StartScan` |

### Action Structure

A ROS2 action contains **three sections**:

#### 1. **Goal**
The parameters of the operation to perform (equivalent to a service request)

#### 2. **Result**
The final result returned at completion (equivalent to a service response)

#### 3. **Feedback**
Information sent **periodically** during execution (e.g., progress percentage)

### Example: StartScan.action

**Reading Exercise:**
> "Open the file `/home/ubuntu/ros2_ws/src/jerro/jerro_msgs/action/StartScan.action` and identify the three sections."

In [ ]:
# Let's visualize the structure of StartScan.action
!cat /home/ubuntu/ros2_ws/src/jerro/jerro_msgs/action/StartScan.action

### Analysis of StartScan.action

**Goal:**
- `start`: boolean to start the scan
- **New filtering parameters**: `enable_filtering`, `enable_statistical_filter`, etc.

**Result:**
- `point_cloud`: The resulting 3D point cloud
- `success`: Boolean indicating whether the scan succeeded
- `message`: Descriptive message

**Feedback (Periodic update):**
- `progress`: Float between 0.0 and 1.0 indicating scan progress

### Sequence Diagram

```
Client                    Action Server
  |                             |
  |──────(Goal)───────────────> |
  |                             | goal_callback()
  | <───(Accept/Reject)────────|
  |                             | execute_callback()
  | <───(Feedback 10%)─────────|
  |                             |
  | <───(Feedback 50%)─────────|
  |                             |
  | <───(Feedback 90%)─────────|
  |                             |
  | <───(Result + Success)─────|
  |                             |
```

---
## Section 3: The StartScan Action in lidar_traitement.py
### Creating an Action Server

In the `lidar_traitement.py` file, an Action Server is created as follows:

```python
from rclpy.action import ActionServer, GoalResponse, CancelResponse
from jerro_msgs.action import StartScan

self._action_server = ActionServer(
    self,                                    # The ROS2 node
    StartScan,                               # Action type
    '/perception/start_scan',                # Topic name
    execute_callback=self.execute_scan_callback,
    goal_callback=self.goal_callback,
    cancel_callback=self.cancel_callback,
    callback_group=self._group,              # See Section 4
)
```

### The Three Callbacks

#### 1. **goal_callback**: Accept or reject the goal

```python
def goal_callback(self, goal_request):
    if not self.set_servo_pos_srv.service_is_ready():
        return GoalResponse.REJECT  # Servo not ready
    if self.actual_scan is None:
        return GoalResponse.REJECT  # No scan received
    return GoalResponse.ACCEPT      # OK, accept it
```

#### 2. **cancel_callback**: Handle cancellation requests

```python
def cancel_callback(self, goal_handle):
    self.get_logger().info("Cancellation requested")
    return CancelResponse.ACCEPT
```

#### 3. **execute_callback**: Execute the action with a loop

This is the core callback. It **starts when the goal is accepted** and **ends when it returns a result**:

```python
def execute_scan_callback(self, goal_handle):
    # Initialization...
    self._scan_state.set_next_state(ScanState.MOVE_TO_START)

    while True:                                    # ← main loop
        match self._scan_state.get_actual_state():
            case ScanState.FINISHING:
                self._finalize_scan()
                result = StartScan.Result()
                result.success = True
                goal_handle.succeed()
                return result                      # ← end of callback (success)

            case ScanState.WAITING_SERVO:
                if prev == ScanState.CANCELLING:
                    goal_handle.canceled()
                    return result                  # ← end of callback (cancellation)
            # ...
```

**Key Points:**
- The callback **loops continuously** until the scan is complete
- It ends with a `return result` (success or cancellation)
- `goal_handle` is available locally (no need to store it as a class attribute)
- Feedback is published directly from the loop: `goal_handle.publish_feedback(feedback)`

### Comprehension Questions

1. **Which line terminates the callback on success?**
   - Answer: `return result` in the `FINISHING` case, after `goal_handle.succeed()`

2. **What happens if `goal_callback` returns `GoalResponse.REJECT`?**
   - Answer: `execute_callback` is never called

3. **Compare with v1: in `lidar_traitement_v1.py`, who drove the state machine?**
   - Answer: A timer called `state_machine_tick()` every 50ms. Here, it is the `while True` loop.

---
## Section 4: Executor and Callback Groups

### What is an Executor?

A ROS2 **executor** is the component responsible for **executing callbacks**: timers, subscribers, services, actions...

When you write `rclpy.spin(node)`, you are implicitly using a **single-threaded** executor that processes all callbacks **one after another**.

### The Problem: execute_callback is Blocking

`execute_scan_callback` contains a `while True` loop that **only ends at the completion of the scan** (several seconds). Meanwhile, the thread is occupied.

However, during this `while True`, the loop waits for `self._servo_done` to become `True`:

```python
case ScanState.WAITING_SERVO:
    if not self._servo_done:   # ← waits here (busy-wait)
        continue
```

Who sets `_servo_done = True`? It is `_on_servo_response`, a **callback** invoked when the servo confirms its position:

```python
def _on_servo_response(self, _future):
    self._servo_done = True     # ← needs to execute during the while True!
```

### Deadlock with a Single Thread

```
Single thread:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  1. execute_scan_callback() starts, enters while True       │
│     → sends servo command (_send_servo_pos)                 │
│     → WAITING_SERVO: loops waiting for _servo_done...       │
│         while not self._servo_done: continue  ←──BLOCKED!   │
│                                                             │
│  2. _on_servo_response() wants to execute...                │
│     → IMPOSSIBLE! Thread is taken by the while True         │
│                                                             │
│  3. _servo_done will NEVER be set to True                   │
│     → DEADLOCK                                              │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### The Solution: MultiThreadedExecutor

With multiple threads, callbacks can execute in parallel:

```
Thread 1:  execute_scan_callback()  ──[while True... WAITING_SERVO...]──
                                                                         ↑ unblocked!
Thread 2:                    _on_servo_response()  →  _servo_done = True
Thread 3:         scan_callback()  (receives LiDAR data continuously)
```

**Implementation:**

```python
from rclpy.executors import MultiThreadedExecutor

def main(args=None):
    rclpy.init(args=args)
    node = Lidar3D()
    executor = MultiThreadedExecutor()   # ← multiple threads
    executor.add_node(node)
    executor.spin()                      # ← replaces rclpy.spin()
```

### Callback Groups

However, `MultiThreadedExecutor` alone is not enough! ROS2 uses **Callback Groups** to decide which callbacks can run in parallel.

#### MutuallyExclusiveCallbackGroup (Default Behavior)

Callbacks in the **same group** **never execute in parallel**—only one at a time:

```
Group A (MutuallyExclusive):
  callback_1 ──────────────────┐
  callback_2                   └──────────────── (must wait)
```

If `execute_scan_callback` and `cancel_callback` belong to the **same** `MutuallyExclusiveCallbackGroup`, cancellation can never be processed while the scan is running.

#### ReentrantCallbackGroup

Callbacks in the **same group** **can** execute in parallel:

```
Group B (Reentrant):
  execute_callback ──────────────────────────────────
  cancel_callback       ──────                       (can run simultaneously!)
```

**In Practice:**

Each callback created separately (via `create_subscription`, `create_timer`, etc.) receives **its own** `MutuallyExclusiveCallbackGroup` by default. These **different** groups can therefore always run in parallel with a `MultiThreadedExecutor`.

The specific issue in our case is that **multiple action callbacks** (execute + cancel) would belong to the same default group. Placing them in a `ReentrantCallbackGroup` ensures that cancellation can interrupt execution:

```python
from rclpy.callback_groups import ReentrantCallbackGroup

self._group = ReentrantCallbackGroup()

self._action_server = ActionServer(
    self,
    StartScan,
    '/perception/start_scan',
    execute_callback=self.execute_scan_callback,
    goal_callback=self.goal_callback,
    cancel_callback=self.cancel_callback,
    callback_group=self._group,          # ← Reentrant: parallel execution possible
)
```

### Summary Table

| | `rclpy.spin()` | `MultiThreadedExecutor` alone | `MultiThreadedExecutor` + `ReentrantCallbackGroup` |
|-|----------------|------------------------------|------------------------------------------------------|
| **Threads** | 1 | N | N |
| **Blocking while True** | ❌ Deadlock | ✅ OK | ✅ OK |
| **Cancel during scan** | ❌ Impossible | ⚠️ Partial | ✅ Guaranteed |
| **scan_callback in parallel** | ❌ | ✅ | ✅ |

### Comprehension Questions

1. **If using `rclpy.spin()` with a while True loop, what happens during WAITING_SERVO?**
   - Answer: Deadlock—the while True loop monopolizes the thread, `_on_servo_response` can never run, and `_servo_done` remains False forever.

2. **Without `ReentrantCallbackGroup`, what would happen if you send a cancellation request during a scan?**
   - Answer: `cancel_callback` could not execute while `execute_scan_callback` was running (same `MutuallyExclusive` group).

3. **`lidar_traitement.py` used `rclpy.spin()`. Why didn't it deadlock?**
   - Answer: The state machine was inside a timer—each call to `state_machine_tick()` completed in a few milliseconds. No blocking loop. The thread was always released quickly.

---
## Section 5: State Machine inside the while True Loop

### Architecture: Timer vs while True

In `lidar_traitement_v1.py`, the state machine was driven by a **timer**:

```
Timer (every 50ms) ──> state_machine_tick()  ──> advances one state
                     ↑___________________________|  (short callback)
```

In `lidar_traitement.py`, it is integrated directly into the **action's while loop**:

```
execute_scan_callback() ─────────────────────────────────────────────>
   while True:                                                         |
     MOVE_TO_START → WAITING_SERVO → SCANNING → ... → FINISHING        |
                                                         return result |
```

**Advantage:** All scan logic is in a single place. More readable, more direct.

### Current ScanState States

```python
class ScanState:
    IDLE = 0          # Not used in the loop (initial state)
    MOVE_TO_START = 1 # Move servo to start position
    WAITING_SERVO = 2 # Wait for servo response (busy-wait)
    SCANNING = 3      # Scan and publish feedback
    RESETTING = 4     # Return servo to neutral position
    FINISHING = 5     # Finalize, publish cloud, return result
    CANCELLING = 6    # Cancellation in progress
```

### Transition Diagram

```
(goal received) → MOVE_TO_START
                   ↓ (_send_servo_pos)
              WAITING_SERVO ←────────────────────────────┐
                   ↓ (_servo_done = True)                │
               SCANNING ──(cancel)──> CANCELLING         │
                   │                      │ (_send_servo_pos(1500))
                   │ (end of travel)       └──────────────┘
                   ↓
               RESETTING
                   ↓ (_send_servo_pos(1500))
              WAITING_SERVO
                   ↓
               FINISHING
                   ↓
             return result
```

### Adding the PROCESSING State

To add filtering, a `PROCESSING` state must be inserted between `WAITING_SERVO` (after RESETTING) and `FINISHING`:

```
... RESETTING
       ↓
  WAITING_SERVO
       ↓ (if enable_filtering)
  PROCESSING          ↓ (otherwise)
       └────────────> FINISHING
                          ↓
                    return result
```

**Question:** How to conditionally transition to PROCESSING or directly to FINISHING?

**Hint:** Use `goal_handle.request.enable_filtering`

**Caution:** In the version using `while True`, `goal_handle` is available as a **local variable** of the callback (not as a class attribute). You can use it directly inside the loop.

---
## Section 6: Open3D Filters - Theory

### Introduction to Filtering

#### Why Filter Point Clouds?

Raw point clouds from a LiDAR often contain:
- **Noise**: Erroneous points due to sensor limitations
- **Outliers**: Stray points far away from their neighborhood
- **Excessive density**: Too many points for real-time processing
- **Data outside the region of interest**: Points beyond the robot's immediate horizon

#### Sources of Noise in a LiDAR Scan

- **Multi-path reflections**: The laser bounces multiple times
- **Reflective surfaces**: Mirrors, windows
- **Dust and particles**: Airborne
- **Measurement errors**: Sensor inaccuracy
- **Mechanical vibrations**: From the servo or robot

### Filter 1: Crop (Bounding Box)

#### Principle

Trims the cloud to keep only points **inside a defined volume** (bounding box).

Useful for:
- Limiting the region of interest (e.g., 5m around the robot)
- Eliminating the floor or ceiling
- Reducing the volume of data to process

#### Parameters

- `x_min`, `x_max`: Limits in X (front/back of the robot)
- `y_min`, `y_max`: Limits in Y (left/right)
- `z_min`, `z_max`: Limits in Z (bottom/top)

#### Visualization

```
         z ↑
           │    crop_z_max
           │   ┌───────────┐
           │   │           │
           │   │   Zone    │
           │   │ preserved │
           │   │           │
           │   └───────────┘
           │    crop_z_min
           └──────────────────> x
```

#### Open3D Method

```python
bbox = o3d.geometry.AxisAlignedBoundingBox(
    min_bound=[x_min, y_min, z_min],
    max_bound=[x_max, y_max, z_max]
)
cropped_cloud = cloud.crop(bbox)
```

#### Reference

**Open3D Documentation:**  
https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#bounding-volume-cropping

#### Questions

1. **If the robot is 0.3m tall, what value of `z_min` allows eliminating the floor?**
   - Answer: Around `z_min = 0.05` to `0.1` to avoid clipping the robot itself

2. **Why apply crop first?**
   - Answer: To reduce data volume **before** other expensive filters

### Filter 2: Statistical Outlier Removal

#### Principle

Removes **statistically aberrant** points based on their distance to neighbors.

**Algorithm:**
1. For each point, compute the mean distance to its **K nearest neighbors**
2. Compute the **mean** and **standard deviation** of all these distances
3. Remove points whose mean distance > `mean + std_ratio × standard_deviation`

#### Visualization

```
    ●                     ●  ← Isolated outlier (will be removed)
    
         ● ● ●
        ● ● ● ●           ← Dense cluster (will be kept)
         ● ● ●
    
                 ●        ← Outlier (will be removed)
```

#### Parameters

- **`nb_neighbors`**: Number of neighbors to consider (e.g., 20)
  - Higher → more robust but slower
  - Lower → faster but less reliable

- **`std_ratio`**: Standard deviation multiplier threshold (e.g., 2.0)
  - Higher → **more permissive** (keeps more points)
  - Lower → **stricter** (removes more points)

#### Open3D Method

```python
filtered_cloud, indices = cloud.remove_statistical_outlier(
    nb_neighbors=20,
    std_ratio=2.0
)
# filtered_cloud: filtered cloud
# indices: list of indices of kept points
```

#### Reference

**Open3D Documentation:**  
https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#statistical-outlier-removal

#### Questions

1. **Will a higher `std_ratio` make the filter stricter or more permissive?**
   - Answer: **More permissive** (keeps more points because the threshold is higher)

2. **Why might a `nb_neighbors` that is too small risk eliminating valid points?**
   - Answer: With few neighbors, the mean distance is less statistically reliable

3. **When should this filter be used?**
   - Answer: To eliminate **noise** and aberrant **isolated points**

### Filter 3: Voxel Downsampling

#### Principle

Reduces point cloud **density** using a 3D grid of **voxels** (cubes).

**Algorithm:**
1. Divide space into a 3D grid of fixed-size voxels
2. For each voxel containing points:
   - Compute the **centroid** (mean) of the points inside the voxel
   - Replace all points with this single centroid

**Result:** Maximum one point per voxel

#### 2D Visualization

```
Before:                  After:
┌────┬────┬────┐        ┌────┬────┬────┐
│ ●● │ ●  │    │        │ ●  │ ●  │    │
│ ●  │    │  ● │   →    │    │    │  ● │
├────┼────┼────┤        ├────┼────┼────┤
│ ●●●│ ●● │    │        │ ●  │ ●  │    │
│ ●  │ ●  │    │        │    │    │    │
└────┴────┴────┘        └────┴────┴────┘

8 points → 4 points
```

#### Parameters

- **`voxel_size`**: Voxel size in meters (e.g., 0.02 = 2 cm)
  - Larger → **fewer points** (higher reduction)
  - Smaller → **more points** (lower reduction)

#### Open3D Method

```python
downsampled_cloud = cloud.voxel_down_sample(voxel_size=0.02)
```

#### Reference

**Open3D Documentation:**  
https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#downsampling

#### Questions

1. **Will a `voxel_size` of 0.05m instead of 0.02m result in more or fewer points?**
   - Answer: **Fewer points** (larger voxels = fewer voxels = fewer points)

2. **What is the advantage of voxel downsampling over simple random sampling?**
   - Answer: Preserves a **uniform spatial distribution**, no empty patches

3. **When should this filter be used?**
   - Answer: To reduce **density** and accelerate subsequent processing

### Filter Application Order

The order of filters is **important** for efficiency and quality of the result.

#### Recommended Order

1. **Crop** first
   - Why? Reduces data volume **before** costly filters
   - Example: 100,000 points → 50,000 points

2. **Statistical Outlier Removal** second
   - Why? Eliminates noise on already reduced data
   - Example: 50,000 points → 45,000 points

3. **Voxel Downsampling** last
   - Why? Optimizes final density after cleaning
   - Example: 45,000 points → 10,000 points

#### Filtering Pipeline

```
Raw cloud (100k points, noisy, wide area)
    ↓
[CROP] → Limits to region of interest
    ↓
Cropped cloud (50k points, noisy, reduced area)
    ↓
[STATISTICAL] → Eliminates outliers
    ↓
Cleaned cloud (45k points, clean, reduced area)
    ↓
[VOXEL] → Reduces density
    ↓
Final cloud (10k points, clean, optimized)
```

#### Summary Table

| Filter | Objective | Reduces points? | Computational cost |
|--------|----------|-----------------|--------------------|
| Crop | Limit region | Yes (a lot) | Low |
| Statistical | Eliminate noise | Yes (moderate) | High |
| Voxel | Reduce density | Yes (a lot) | Moderate |

---
## Section 7: Practical Exercises

### Environment Preparation

Before starting, update your workspace with the new action definition:

In [ ]:
%%bash
cd ~/ros2_ws/src/jerro
git pull
cd ~/ros2_ws
colcon build --packages-select jerro_msgs
source install/setup.bash

### Exercise 1: Understanding the New Action Definition

**Objective:** Analyze the modified `StartScan.action` definition

**Instructions:**
1. Open the `StartScan.action` file
2. Identify all new parameters added to the Goal
3. For each parameter, explain its role using the Open3D documentation

**Questions:**
- Why are some parameters `bool` and others `float32`?
- How many parameters in total were added?
- What is the suggested default value for `voxel_size`?

### Exercise 2: Replacing the Timer with while True (30 min)

**Objective:** Transform `execute_scan_callback` to directly contain the state machine

**File to modify:** `lidar_traitement.py`

**Context:** The current version of the file still contains the commented-out timer. Your task is to rewrite `execute_scan_callback` with the `while True` loop.

**TODO 1: Configure ReentrantCallbackGroup**

```python
# TODO: Import ReentrantCallbackGroup
from rclpy.callback_groups import ___

# TODO: Create the group in __init__
self._group = ___

# TODO: Assign it to the action server
self._action_server = ActionServer(
    ...
    callback_group=___,
)
```

**TODO 2: Rewrite execute_scan_callback**

```python
def execute_scan_callback(self, goal_handle):  # ← no more async
    # TODO: Initialize scan (servo_pos, global_scan_3d, servo_done, initial state)

    while True:
        match self._scan_state.get_actual_state():

            case ScanState.MOVE_TO_START:
                # TODO: Send servo to start position

            case ScanState.WAITING_SERVO:
                # TODO: Busy-wait if _servo_done is False
                # TODO: Depending on previous state, transition to SCANNING, FINISHING, or handle cancellation

            case ScanState.SCANNING:
                # TODO: Check if cancellation requested (goal_handle.is_cancel_requested)
                # TODO: Collect scan and publish feedback
                # TODO: Decrement servo position and send if necessary

            case ScanState.RESETTING:
                # TODO: Send servo to neutral position (1500)

            case ScanState.CANCELLING:
                # TODO: Send servo to neutral position

            case ScanState.FINISHING:
                # TODO: Finalize scan, create result, call goal_handle.succeed()
                # TODO: return result  ← terminates callback
```

**TODO 3: Use MultiThreadedExecutor in main()**

```python
def main(args=None):
    rclpy.init(args=args)
    node = Lidar3D()
    # TODO: Create a MultiThreadedExecutor, add node, call spin()
```

**Hints:**
- `goal_handle` is available as a local variable inside `execute_scan_callback`
- For busy-wait: `if not self._servo_done: continue`
- The callback ends with `return result` (no Future or `set_result`)
- Do not forget to reset `_servo_done = False` before sending a servo command

### Exercise 3: Adding the PROCESSING State

**Objective:** Integrate a new state into the state machine

**File to modify:** `/home/ubuntu/ros2_ws/src/jerro_perception/jerro_perception/lidar_traitement.py`

**TODO 1: Modify the ScanState class**

```python
class ScanState:
    IDLE = 0
    MOVE_TO_START = 1
    WAITING_SERVO = 2
    SCANNING = 3
    RESETTING = 4
    # TODO: Add PROCESSING state = ___
    FINISHING = ___  # TODO: Adjust value
    CANCELLING = ___  # TODO: Adjust value
```

**TODO 2: Add the PROCESSING case in state_machine_tick()**

```python
case ScanState.PROCESSING:
    # TODO: Call self._process_point_cloud()
    # TODO: Transition to FINISHING state
    pass
```

**TODO 3: Modify the WAITING_SERVO case for conditional transition**

```python
case ScanState.WAITING_SERVO:
    if not self._servo_done:
        return
    prev = self._scan_state.get_previous_state()
    if prev == ScanState.RESETTING:
        # TODO: Check if filtering is enabled in the goal
        # TODO: If yes, transition to PROCESSING, otherwise to FINISHING
        pass
```

**Hint:** Use `self._goal_handle.request.enable_filtering`

### Exercise 4: Implementing Filters

**Objective:** Create the three filtering methods

**File to modify:** `lidar_traitement.py`

#### Filter 1: _apply_crop

```python
def _apply_crop(self, cloud, x_range, y_range, z_range):
    """
    TODO: Crops the cloud within a bounding box

    Args:
        cloud: Open3D cloud
        x_range: [min, max] in x
        y_range: [min, max] in y
        z_range: [min, max] in z

    Returns:
        Cropped cloud

    Reference: [https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#bounding-volume-cropping](https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#bounding-volume-cropping)

    Hints:
    - Create an AxisAlignedBoundingBox with min_bound and max_bound
    - Use the cloud's .crop() method
    - Log the point count before/after
    """
    # TODO: Your code here
    pass
```

#### Filter 2: _apply_statistical_filter

```python
def _apply_statistical_filter(self, cloud, nb_neighbors, std_ratio):
    """
    TODO: Statistical filter to eliminate outliers

    Args:
        cloud: Open3D cloud
        nb_neighbors: Number of neighbors to consider
        std_ratio: Standard deviation ratio

    Returns:
        Filtered cloud

    Reference: [https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#statistical-outlier-removal](https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#statistical-outlier-removal)

    Hints:
    - Use the .remove_statistical_outlier() method
    - This method returns TWO values: (filtered_cloud, indices)
    - Log the point count before/after
    """
    # TODO: Your code here
    pass
```

#### Filter 3: _apply_voxel_downsampling

```python
def _apply_voxel_downsampling(self, cloud, voxel_size):
    """
    TODO: Downsamples the cloud using a voxel grid

    Args:
        cloud: Open3D cloud
        voxel_size: Voxel size in meters

    Returns:
        Downsampled cloud

    Reference: [https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#downsampling](https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#downsampling)

    Hints:
    - Use the .voxel_down_sample() method
    - Log the point count before/after
    """
    # TODO: Your code here
    pass
```

### Exercise 5: Integration in _process_point_cloud()

**Objective:** Create the method that orchestrates all filters

```python
def _process_point_cloud(self):
    """
    TODO: Apply the filters configured in the goal to the global cloud

    Application order:
    1. Crop (reduces data volume)
    2. Statistical outlier removal (eliminates noise)
    3. Voxel downsampling (optimizes final density)

    Each filter should only be applied if its activation boolean is True
    """
    if self._goal_handle is None:
        return

    goal = self._goal_handle.request
    self.get_logger().info("Starting point cloud processing...")

    filtered_cloud = self.global_scan_3d

    # TODO 1: Apply crop if goal.enable_crop is True
    # Hints:
    #   - Create ranges: x_range = [goal.crop_x_min, goal.crop_x_max]
    #   - Call self._apply_crop(filtered_cloud, x_range, y_range, z_range)
    #   - Assign result to filtered_cloud

    # TODO 2: Apply statistical filter if goal.enable_statistical_filter is True
    # Hints:
    #   - Retrieve parameters: goal.statistical_nb_neighbors, goal.statistical_std_ratio
    #   - Call self._apply_statistical_filter(...)

    # TODO 3: Apply voxel downsampling if goal.enable_voxel_downsampling is True
    # Hints:
    #   - Retrieve parameter: goal.voxel_size
    #   - Call self._apply_voxel_downsampling(...)

    # Replace global cloud with filtered cloud
    self.global_scan_3d = filtered_cloud
    self.get_logger().info("Point cloud processing completed")
```

### Exercise 6: Testing and Tuning

**Objective:** Test your implementation and find optimal parameters

#### Step 1: Compilation

In [ ]:
%%bash
cd ~/ros2_ws
colcon build --packages-select jerro_perception
source install/setup.bash

#### Step 2: Run the node

```bash
ros2 run jerro_perception lidar_traitement
```

#### Step 3: Send a goal with filtering enabled

**Terminal 2:**
```bash
ros2 action send_goal /perception/start_scan jerro_msgs/action/StartScan \
  "{
    start: true,
    enable_filtering: true,
    enable_crop: true,
    crop_x_min: -2.5, crop_x_max: 2.5,
    crop_y_min: -2.5, crop_y_max: 2.5,
    crop_z_min: -0.5, crop_z_max: 1.0,
    enable_statistical_filter: true,
    statistical_nb_neighbors: 20,
    statistical_std_ratio: 2.0,
    enable_voxel_downsampling: true,
    voxel_size: 0.02
  }" \
  --feedback
```

#### Step 4: Visualization in RViz

**Terminal 3:**
```bash
rviz2
```

- Add a `PointCloud2` display
- Topic: `/perception/lidar_pc2`
- Fixed Frame: `base_link`

#### Step 5: Experimentation

Test different configurations:

1. **Without filtering**: `enable_filtering: false`
2. **Crop only**: Only enable_crop
3. **Statistical only**: Only enable_statistical_filter
4. **All filters**: All enabled
5. **Aggressive parameters**: `voxel_size: 0.05`, `std_ratio: 1.0`

**Analysis Questions:**
- How many points before/after filtering?
- Which filter reduces the point count the most?
- Which parameters give the best quality/density tradeoff for your environment?

---
## Section 8: Laboratory 2 Synthesis Questions

Answer the following questions to validate your understanding:

### Question 1: Actions vs Services

**Why use an action instead of a service for StartScan?**


---

### Question 2: Executor and Callback Groups

**Explain why `lidar_traitement.py` needs both `MultiThreadedExecutor` AND `ReentrantCallbackGroup`.**

Your answer must mention:
- What would happen without `MultiThreadedExecutor` (deadlock in WAITING_SERVO)
- What would happen without `ReentrantCallbackGroup` (cancellation impossible)
- The role of each

---

### Question 3: Parameterization

**What is the advantage of having filtering parameters in the action Goal rather than as [ROS2 parameters](https://docs.ros.org/en/rolling/Tutorials/Beginner-CLI-Tools/Understanding-ROS2-Parameters/Understanding-ROS2-Parameters.html) of the node?**

---

### Question 4

In the file `/home/ubuntu/ros2_ws/src/jerro/jerro_description/urdf/jerro.urdf`, the axes of the joint `base_link_to_servomotor` are x=0, y=-1, and z=0. Based on the URDF file and the [LiDAR datasheet](https://download-en.slamtec.com/api/download/rplidar-s2m1-Rx-datasheet/2.3?lang=en), explain these values.

```xml
 <joint name="base_link_to_servomotor" type="revolute">
    <parent link="base_link" />
    <child link="servomotor" />
    <origin xyz="0.088 0 0.054" rpy="0 0 0" />
    <axis xyz="0 -1 0"/>
    <limit effort="1000.0" lower="-1" upper="1" velocity="0.5"/>  
    </joint>
```



---

### Question 5: Filter Order

**Explain why crop is applied first and voxel downsampling last.**



---

### Question 6: Statistical Parameters

**Why might a `nb_neighbors` that is too small in the statistical filter risk eliminating valid points?**


---

### Question 7

**Implement the 3 filters as presented. Present images to show the difference between filters**

---

### Question 8

**To use the camera with the Raspberry Pi, modifications to certain files had to be made. Based on the `v4l2_camera` package readme [v4l2_camera](https://docs.ros.org/en/rolling/p/v4l2_camera/), explain what these modifications were and why.**

---

### Question 9

**By observing the messages of the topics `image_raw` and `image_raw/compressed`, explain the difference between these two message types and what their usage implies**

---

### Question 10
**To use the camera, a configuration file was placed in the directory `/home/ubuntu/.ros/camera_info/ost.yaml`. Explain the various parameters of this file and what its utility is. Also, what would be the consequences of its absence**


---
## Demonstration to the Lab Instructor

At the end of this laboratory, you must demonstrate:

### Validation Checklist

- [ ] **Code compiled without errors**
- [ ] **PROCESSING state added** to the state machine
- [ ] **Three filters implemented** (crop, statistical, voxel)
- [ ] **Conditional transition** works (filtering enabled/disabled)
- [ ] **Logs display** the number of points before/after each filter
- [ ] **RViz visualization** shows the difference before/after filtering
- [ ] **Parameters optimized** for your environment
- [ ] **Synthesis questions** answered

### Bonus (Optional)

- [ ] **Radius outlier removal** implemented and tested
- [ ] **Quantitative comparison** of different configurations (results table)
- [ ] **Performance analysis** (execution time of each filter)

---
## Useful Resources

### Open3D Documentation

- **Main page**: https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html
- **Statistical Outlier Removal**: https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#statistical-outlier-removal
- **Radius Outlier Removal**: https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#radius-outlier-removal
- **Downsampling**: https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#downsampling
- **Bounding Volume Cropping**: https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#bounding-volume-cropping

### ROS2 Documentation

- **Actions**: https://docs.ros.org/en/rolling/Tutorials/Beginner-CLI-Tools/Understanding-ROS2-Actions/Understanding-ROS2-Actions.html
- **Action Servers (Python)**: https://docs.ros.org/en/rolling/Tutorials/Intermediate/Writing-an-Action-Server-Client/Py.html
- **Executors and Callback Groups**: https://docs.ros.org/en/rolling/Concepts/Intermediate/About-Executors.html

### Reference Code

- `lidar_traitement.py`: Version with Action Server, `while True`, `MultiThreadedExecutor`, `ReentrantCallbackGroup`
- `lidar_traitement_v1.py`: Previous version with services and timer (for comparison)